<a href="https://colab.research.google.com/github/DhrumilPrajapati03/Linkedin-post-generator/blob/main/colab_linkedin_post_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LinkedIn Post Generator

This notebook implements a simple LinkedIn post generator using Groq's LLM API via LangChain and Streamlit for the user interface.

## Setup Instructions
1. Get a Groq API key from [https://console.groq.com/keys](https://console.groq.com/keys)
2. Run all cells in order
3. Access the Streamlit app through the public URL that will be generated

In [27]:
# Install required dependencies
!pip install langchain langchain-groq streamlit streamlit-extras pyngrok

In [28]:
# Set up the API keys
import os
from getpass import getpass

# Prompt for the Groq API key if not already set
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

# Prompt for the ngrok authtoken
ngrok_token = getpass("Enter your ngrok authtoken (from https://dashboard.ngrok.com/get-started/your-authtoken): ")

Enter your ngrok authtoken (from https://dashboard.ngrok.com/get-started/your-authtoken): ··········


In [29]:
# Configure ngrok with the authtoken
from pyngrok import ngrok, conf

# Set the ngrok auth token
conf.get_default().auth_token = ngrok_token
print("ngrok authentication configured successfully!")

ngrok authentication configured successfully!


In [30]:
# Create the llm_helper.py code

%%writefile llm_helper.py
from langchain_groq import ChatGroq
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import os

def get_llm():
    """Factory function to create new LLM instances when needed"""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY not found in environment variables")

    return ChatGroq(
        api_key=api_key,
        model_name="llama3-70b-8192"  # or whichever model you want to use
    )

def llm(template, **kwargs):
    """Function to generate content using LLM with the given template and variables"""
    llm_instance = get_llm()  # Get a fresh instance every time

    prompt = PromptTemplate.from_template(template)
    chain = LLMChain(llm=llm_instance, prompt=prompt)

    return chain.run(**kwargs)

Overwriting llm_helper.py


In [34]:
# Create the post_generator.py code

%%writefile post_generator.py
from llm_helper import llm

def generate_post(topic, tone, audience, cta=None):
    """
    Generate a LinkedIn post based on the given parameters

    Args:
        topic: The topic for the LinkedIn post
        tone: The tone of the post (professional, casual, etc.)
        audience: The target audience for the post
        cta: Call to action (optional)

    Returns:
        str: Generated LinkedIn post
    """
    template = """
    You are a professional LinkedIn content creator. Create a post about {topic}.
    The tone should be {tone}.
    The target audience is {audience}.

    {cta_instruction}

    The post should be well-structured, engaging, and suitable for LinkedIn.
    Ensure it's not too long (maximum 5500 characters).

    OUTPUT:
    """

    # Add CTA instruction if provided
    cta_instruction = f"Include this call to action: {cta}" if cta else "No specific call to action is required."

    # Pass the variables to the template
    return llm(
        template=template,
        topic=topic,
        tone=tone,
        audience=audience,
        cta_instruction=cta_instruction
    )

Overwriting post_generator.py


In [35]:
# Create the app.py file for Streamlit

%%writefile app.py
import streamlit as st
import time
from post_generator import generate_post

st.set_page_config(page_title="LinkedIn Post Generator", layout="wide")

# Add some custom CSS for better appearance
st.markdown("""
<style>
    .main .block-container {
        padding-top: 2rem;
    }
    .stTextArea textarea {
        height: 200px;
    }
    .result-area {
        background-color: #f0f2f6;
        border-radius: 10px;
        padding: 20px;
        margin-top: 20px;
    }
</style>
""", unsafe_allow_html=True)

st.title("LinkedIn Post Generator")
st.markdown("Generate engaging LinkedIn posts with AI in seconds!")

with st.form(key="post_form"):
    col1, col2 = st.columns(2)

    with col1:
        topic = st.text_area("Topic", placeholder="Enter the topic for your post")
        tone = st.selectbox(
            "Tone",
            options=["Professional", "Casual", "Inspirational", "Educational", "Humorous"]
        )

    with col2:
        audience = st.text_input("Target Audience", placeholder="Who is your target audience?")
        cta = st.text_input("Call to Action (Optional)", placeholder="Add a call to action")

    submit_button = st.form_submit_button(label="Generate Post", use_container_width=True)

if submit_button:
    if not topic or not audience:
        st.error("Please fill in the required fields.")
    else:
        with st.spinner("Generating your LinkedIn post..."):
            # Generate the post
            post = generate_post(topic=topic, tone=tone, audience=audience, cta=cta if cta else None)

            # Display the result
            st.subheader("Your LinkedIn Post:")
            st.markdown("""
<style>
    .main .block-container {
        padding-top: 2rem;
    }
    .stTextArea textarea {
        height: 200px;
    }
    .result-area {
        background-color: #f0f2f6;
        border-radius: 10px;
        padding: 20px;
        margin-top: 20px;
        color: black;
    }
</style>
""", unsafe_allow_html=True)

            # Add a copy button
            st.text_area("Copy your post", value=post, height=200, key="result")

            st.success("Your LinkedIn post has been generated successfully!")

st.markdown("---")
st.markdown("Created with ❤️ using LangChain and Groq")

Overwriting app.py


In [36]:
# Run the Streamlit app using ngrok for public access
from pyngrok import ngrok
import subprocess
import time

# Check if Streamlit is already running (from the previous cell)
try:
    if 'streamlit_process' not in locals():
        # Start Streamlit in the background
        streamlit_process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port=8501'], stdout=subprocess.PIPE)
        # Wait for Streamlit to start
        time.sleep(5)
except:
    # Start Streamlit in the background
    streamlit_process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port=8501'], stdout=subprocess.PIPE)
    # Wait for Streamlit to start
    time.sleep(5)

# Create a public URL using ngrok
public_url = ngrok.connect(addr="8501").public_url
print(f"\nStreamlit app is running at: {public_url}\n")
print("This URL will be active as long as this notebook is running.")
print("To stop the app, interrupt the kernel (click the ⏹️ button in the toolbar).")


Streamlit app is running at: https://7713-34-80-160-189.ngrok-free.app

This URL will be active as long as this notebook is running.
To stop the app, interrupt the kernel (click the ⏹️ button in the toolbar).
